# Twenty horizontal BM4 recurrence trajectories: calculation

This notebook is the compute-only half of the horizontal recurrence study. It defines the complete reproducible campaign, runs 20 independent `BM4Implicit` trajectories in parallel, and writes the dense trajectories and worker diagnostics to `results.npz`.

It intentionally contains no plotting or animation code. Run this notebook on the compute environment; then copy the directory, or at least `results.npz`, to the environment where `visualization.ipynb` will be used.

## Reproducible campaign and output contract

The 20 initial positions are equally spaced on the horizontal half-line from the periodic-cell centre toward the right boundary. Every trajectory covers 50 normalized cycles with 80 complete BM4 steps per cycle and **20 saved states per cycle**. The complete archive therefore contains 1,001 aligned states per trajectory at intervals of $0.05$.

The compressed NPZ stores all positions, saved times, nonlinear-work summaries, wall time, numerical configuration, potential specification, and initial-condition geometry. During integration, the parent-owned progress stream is shown live in the cell and mirrored to `calculation.log` for monitoring with `tail -f`.

In [1]:
from contextlib import redirect_stderr
from pathlib import Path
import os
import sys

from IPython.display import Markdown, display
import numpy as np

from diagnostics import write_parallel_bm4_recurrence_npz
from diagnostics.paths import find_project_root
from initial_conditions import GCInitialConfiguration
from potential import load_gc2d_h5_potential
from studies import domain_center
from studies.bm4_parallel_recurrence import (
    ParallelBM4RecurrenceConfig,
    run_parallel_bm4_recurrence,
)

In [2]:
# Colocated notebooks, result archive, and live calculation log.
project_root = find_project_root(Path.cwd())
notebook_directory = (
    project_root
    / "notebooks/developements/recurrences/study_20_horizontal_bm4_recurrences"
)
notebook_directory.mkdir(parents=True, exist_ok=True)
results_path = notebook_directory / "results.npz"
execution_log_path = notebook_directory / "calculation.log"

# Measured, nondimensionalized GC2D potential.
data_path = project_root / "data/potential/V1/PHI_2.h5"
magnetic_field = 1.5
characteristic_length = 0.06
mode_selection = (0, 1)
interpolation_order = 3

# Twenty half-open horizontal positions from the cell centre to the right edge.
particle_count = 20

# Fifty cycles with 80 BM4 steps and 20 stored states per cycle.
cycle_count = 50
steps_per_cycle = 80
saved_samples_per_cycle = 20
t_span = (0.0, float(cycle_count))
rho = 0.3
coupling_frequency = float(np.pi / 8.0)
newton_absolute_tolerance = 1e-12
newton_relative_tolerance = 1e-11
newton_max_iterations = 40
jacobian_relative_step = float(np.cbrt(np.finfo(float).eps))

available_cpu_count = os.cpu_count() or 1
worker_count = min(available_cpu_count, particle_count)
recurrence_tolerance_fraction = 0.01

if not data_path.is_file():
    raise FileNotFoundError(f"Measured HDF5 potential not found: {data_path}")
potential = load_gc2d_h5_potential(
    data_path,
    B=magnetic_field,
    characteristic_length=characteristic_length,
    indx=mode_selection,
    interpolation_order=interpolation_order,
)
center_x, center_y = domain_center(potential)
horizontal_spacing = potential.grid.period / (2.0 * particle_count)
initial_x = center_x + horizontal_spacing * np.arange(particle_count)
initial_y = np.full(particle_count, center_y)
initial_configuration = GCInitialConfiguration.from_components(x=initial_x, y=initial_y)

config = ParallelBM4RecurrenceConfig(
    particle_count=particle_count,
    t_span=t_span,
    steps_per_cycle=steps_per_cycle,
    saved_samples_per_cycle=saved_samples_per_cycle,
    rho=rho,
    coupling_frequency=coupling_frequency,
    absolute_tolerance=newton_absolute_tolerance,
    relative_tolerance=newton_relative_tolerance,
    max_iterations=newton_max_iterations,
    jacobian_relative_step=jacobian_relative_step,
    worker_count=worker_count,
    progress=True,
)

saved_sample_interval = 1.0 / saved_samples_per_cycle
integration_steps_per_saved_sample = steps_per_cycle // saved_samples_per_cycle
assert config.cycle_count == 50
assert config.step_count == 4_000
assert config.output_sample_count == 1_001
assert config.integration_step == 0.0125
assert saved_sample_interval == 0.05
assert steps_per_cycle % saved_samples_per_cycle == 0
assert integration_steps_per_saved_sample == 4
assert config.worker_count == min(available_cpu_count, particle_count)
np.testing.assert_allclose(np.diff(initial_x), horizontal_spacing)
np.testing.assert_allclose(initial_y, center_y)
assert initial_x[0] == center_x
assert initial_x[-1] < potential.grid.xmin + potential.grid.period

display(Markdown(
    f"**Resolved campaign:** `{particle_count}` trajectories, `{config.step_count}` "
    f"BM4 steps per trajectory, `{saved_samples_per_cycle}` saved states per cycle "
    f"(`{config.output_sample_count}` total), and `{config.worker_count}` worker processes."
))

**Resolved campaign:** `20` trajectories, `4000` BM4 steps per trajectory, `20` saved states per cycle (`1001` total), and `8` worker processes.

## Parallel BM4 integration with a persistent live log

With `config.progress=True`, the cell reports completed trajectories, worker count, elapsed time, and ETA after every completion and at least every 30 seconds. The same stream is flushed incrementally to `calculation.log`, so a remote run can be followed from another shell.

In [3]:
class ExecutionLogTee:
    '''Mirror live stderr output to the notebook and a plain-text log file.'''

    def __init__(self, notebook_stream, file_stream):
        self.notebook_stream = notebook_stream
        self.file_stream = file_stream

    def write(self, text):
        self.notebook_stream.write(text)
        self.file_stream.write(text.replace("\r", "\n"))
        self.file_stream.flush()
        return len(text)

    def flush(self):
        self.notebook_stream.flush()
        self.file_stream.flush()


print(f"Live calculation log: {execution_log_path.relative_to(project_root)}", flush=True)
print(
    "From another shell, follow it with "
    f"`tail -f {execution_log_path.relative_to(project_root)}`.",
    flush=True,
)
with execution_log_path.open("w", encoding="utf-8") as log_stream:
    with redirect_stderr(ExecutionLogTee(sys.stderr, log_stream)):
        result = run_parallel_bm4_recurrence(
            potential,
            initial_configuration,
            config=config,
        )

Live calculation log: notebooks/developements/recurrences/study_20_horizontal_bm4_recurrences/calculation.log
From another shell, follow it with `tail -f notebooks/developements/recurrences/study_20_horizontal_bm4_recurrences/calculation.log`.


Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 0.0 s; ETA after first completion.
Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 53.8 s; ETA after first completion.
Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 83.8 s; ETA after first completion.
Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 113.8 s; ETA after first completion.
Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 143.8 s; ETA after first completion.
Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 173.8 s; ETA after first completion.
Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 203.8 s; ETA after first completion.
Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 233.8 s; ETA after first completion.
Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 263.8 s; ETA after first completion.
Parallel BM4 recurrence: 0/20 trajectories; workers 8; elapsed 293.8 s; ETA after first completion.
Para

## Calculation audit

In [4]:
assert result.positions.shape == (particle_count, 2, config.output_sample_count)
assert result.initial_positions.shape == (particle_count, 2)
assert result.runtime_seconds.shape == (particle_count,)
assert result.total_newton_iterations.shape == (particle_count,)

expected_saved_times = np.linspace(t_span[0], t_span[1], config.output_sample_count)
np.testing.assert_allclose(result.times, expected_saved_times)
np.testing.assert_allclose(np.diff(result.times), saved_sample_interval)
np.testing.assert_allclose(result.initial_positions[:, 0], initial_x)
np.testing.assert_allclose(result.initial_positions[:, 1], initial_y)
assert np.all(result.maximum_residual_to_tolerance <= 1.0)

cycle_numbers = np.arange(config.cycle_count + 1, dtype=int)
cycle_boundary_indices = cycle_numbers * saved_samples_per_cycle
np.testing.assert_allclose(
    result.times[cycle_boundary_indices],
    cycle_numbers.astype(float),
)
assert cycle_boundary_indices[-1] == config.output_sample_count - 1

display(Markdown(
    f"All **{particle_count} trajectories** completed in "
    f"**{result.wall_runtime_seconds:.1f} s wall time**. The result contains "
    f"**{result.times.size} saved states per trajectory**, including all "
    f"**{cycle_numbers.size} exact cycle boundaries**."
))

All **20 trajectories** completed in **4366.3 s wall time**. The result contains **1001 saved states per trajectory**, including all **51 exact cycle boundaries**.

## Persist the complete calculation

The archive is deliberately colocated with both notebooks. Re-running this cell atomically replaces the previous result only after the new compressed file has been written successfully.

In [5]:
written_path = write_parallel_bm4_recurrence_npz(
    result,
    results_path,
    metadata={
        "study_name": "Twenty horizontal BM4 recurrence trajectories",
        "potential": {
            "source_path": str(data_path.relative_to(project_root)),
            "magnetic_field": magnetic_field,
            "characteristic_length": characteristic_length,
            "mode_selection": mode_selection,
            "interpolation_order": interpolation_order,
        },
        "initial_conditions": {
            "geometry": "horizontal_half_line_from_cell_center",
            "particle_count": particle_count,
            "center": (center_x, center_y),
            "horizontal_spacing": horizontal_spacing,
            "right_endpoint_excluded": True,
        },
        "recurrence_tolerance_fraction": recurrence_tolerance_fraction,
    },
    overwrite=True,
)
assert written_path.is_file()
display(Markdown(
    f"Wrote **{written_path.relative_to(project_root)}** "
    f"({written_path.stat().st_size / 1024**2:.2f} MiB)."
))

Wrote **notebooks/developements/recurrences/study_20_horizontal_bm4_recurrences/results.npz** (0.29 MiB).